# Validação dos metadados existentes

Aqui vamos responder três perguntas práticas: quais arquivos estão disponíveis, quais colunas existem em cada arquivo e se os valores podem ser lidos corretamente.

O notebook apenas lê os arquivos e mostra os resultados. Nada é renomeado, corrigido ou apagado.

In [1]:
from __future__ import annotations

import csv
import io
import math
import re
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Não foi possível localizar a raiz de app/.')


PROJECT_ROOT = find_project_root(Path.cwd())
TABULAR_SUFFIXES = {'.csv', '.tsv'}
NULL_TOKENS = {'null', 'none', 'nan', 'na', 'n/a'}
EXPECTED_SCHEMA_BY_FILENAME = {
    'annotations.csv': {
        'video_id': 'string',
        'video_name': 'string',
        'class': 'string',
        'user_id': 'string',
        'width': 'number',
        'height': 'number',
        'fps': 'number',
        'url_page': 'url',
        'url_download': 'url',
    },
    'error.csv': {
        'video_id': 'string',
        'video_name': 'string',
        'class': 'string',
        'user_id': 'string',
        'width': 'number',
        'height': 'number',
        'fps': 'number',
        'url_page': 'url',
        'url_download': 'url',
    },
}

# A configuração é uma lista para permitir acrescentar outros datasets sem reescrever o fluxo.
DATASETS = {
    'v-librasil-raw': {
        'root': PROJECT_ROOT / 'data' / 'raw' / 'v-librasil-raw',
        'expected_schema_by_filename': EXPECTED_SCHEMA_BY_FILENAME,
    },
}

{name: str(config['root']) for name, config in DATASETS.items()}

{'v-librasil-raw': 'c:\\Users\\mdbaa\\development\\Dissertacao\\app\\data\\raw\\v-librasil-raw'}

## O que vamos conferir

1. **Arquivos:** nome, caminho, extensão e tamanho de cada arquivo encontrado.
2. **Colunas:** se o cabeçalho tem `video_id`, `video_name`, `class`, `user_id`, `width`, `height`, `fps`, URLs ou alguma coluna inesperada.
3. **Valores:** se existem campos vazios, nulos, números que não podem ser lidos ou URLs inválidas.
4. **Leitura:** se o arquivo usa uma codificação e um formato de CSV que podem ser interpretados.

Os resultados completos ficam nos DataFrames `file_inventory_df`, `schema_inventory_df`, `column_profile_df` e `validation_issues_df`.

In [2]:
def inventory_files(dataset_name: str, root: Path) -> list[dict]:
    if not root.is_dir():
        return []
    rows = []
    for path in sorted(path for path in root.rglob('*') if path.is_file()):
        relative_path = path.relative_to(root).as_posix()
        suffix = path.suffix.lower() or '<sem_extensão>'
        rows.append({
            'dataset': dataset_name,
            'relative_path': relative_path,
            'suffix': suffix,
            'size_bytes': path.stat().st_size,
            'tabular_candidate': suffix in TABULAR_SUFFIXES,
        })
    return rows


def decode_bytes(raw: bytes) -> dict:
    candidates = ['utf-8-sig', 'utf-8', 'utf-16', 'cp1252', 'latin-1']
    for encoding in candidates:
        try:
            text = raw.decode(encoding)
        except UnicodeDecodeError:
            continue
        replacement_count = text.count(chr(0xFFFD))
        control_count = sum(
            ord(char) < 32 and char not in {'\n', '\r', '\t'}
            for char in text
        )
        is_utf8 = encoding in {'utf-8', 'utf-8-sig'}
        status = 'ok' if is_utf8 and replacement_count == 0 and control_count == 0 else 'attention'
        return {
            'text': text,
            'encoding': encoding,
            'encoding_status': status,
            'replacement_count': replacement_count,
            'control_count': control_count,
        }
    return {
        'text': None,
        'encoding': None,
        'encoding_status': 'undecodable',
        'replacement_count': None,
        'control_count': None,
    }


def detect_dialect(text: str, suffix: str) -> tuple[str, csv.Dialect]:
    default_delimiter = '\t' if suffix == '.tsv' else ','
    try:
        dialect = csv.Sniffer().sniff(text[:65536], delimiters=',;\t|')
        return dialect.delimiter, dialect
    except csv.Error:
        dialect = csv.excel_tab if default_delimiter == '\t' else csv.excel
        return default_delimiter, dialect

In [3]:
def is_null(value: str) -> bool:
    return value.strip().casefold() in NULL_TOKENS


def parse_number(value: str) -> float | None:
    try:
        number = float(value.strip())
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def is_valid_url(value: str) -> bool:
    if not value or any(char.isspace() for char in value):
        return False
    parsed = urlparse(value)
    return parsed.scheme in {'http', 'https'} and bool(parsed.netloc)


def infer_value_type(value: str) -> str:
    stripped = value.strip()
    if value == '':
        return 'empty'
    if is_null(value):
        return 'null'
    if is_valid_url(value):
        return 'url'
    if parse_number(value) is not None:
        return 'number'
    if stripped.casefold() in {'true', 'false'}:
        return 'boolean'
    return 'string'


def add_issue(issues: list[dict], file_path: str, row_number: int | None, issue: str, column: str | None = None, value: str | None = None, detail: str | None = None) -> None:
    sample = None if value is None else value[:200]
    issues.append({
        'file_path': file_path,
        'row_number': row_number,
        'column': column,
        'issue': issue,
        'value_sample': sample,
        'detail': detail,
    })


def validate_tabular_file(path: Path, root: Path, expected_schema: dict[str, str] | None) -> tuple[dict, list[dict], list[dict]]:
    relative_path = path.relative_to(root).as_posix()
    expected_schema = expected_schema or {}
    raw = path.read_bytes()
    decoded = decode_bytes(raw)
    issues: list[dict] = []
    if decoded['text'] is None:
        add_issue(issues, relative_path, None, 'encoding_undecodable', detail='Nenhuma codificação suportada pôde ser usada.')
        return ({'file_path': relative_path, 'rows': 0, 'column_names': [], 'expected_columns': list(expected_schema), 'missing_columns': list(expected_schema), 'unexpected_columns': [], 'duplicate_columns': [], 'blank_column_names': [], 'encoding': None, 'encoding_status': 'undecodable', 'delimiter': None, 'row_width_errors': 0, 'status': 'attention'}, [], issues)

    text = decoded['text']
    delimiter, dialect = detect_dialect(text, path.suffix.lower())
    try:
        rows = list(csv.reader(io.StringIO(text), dialect))
    except csv.Error as error:
        add_issue(issues, relative_path, None, 'csv_parse_error', detail=str(error))
        rows = []

    if not rows:
        add_issue(issues, relative_path, None, 'empty_tabular_file')
        return ({'file_path': relative_path, 'rows': 0, 'column_names': [], 'expected_columns': list(expected_schema), 'missing_columns': list(expected_schema), 'unexpected_columns': [], 'duplicate_columns': [], 'blank_column_names': [], 'encoding': decoded['encoding'], 'encoding_status': decoded['encoding_status'], 'delimiter': delimiter, 'row_width_errors': 0, 'status': 'attention'}, [], issues)

    header = rows[0]
    data_rows = rows[1:]
    duplicate_columns = sorted(column for column, count in Counter(header).items() if column and count > 1)
    blank_column_names = [f'posição_{index}' for index, column in enumerate(header, start=1) if not column.strip()]
    missing_columns = [column for column in expected_schema if column not in header]
    unexpected_columns = [column for column in header if column not in expected_schema]
    for column in duplicate_columns:
        add_issue(issues, relative_path, 1, 'duplicate_column', column=column)
    for position in [index for index, column in enumerate(header, start=1) if not column.strip()]:
        add_issue(issues, relative_path, 1, 'blank_column_name', column=f'posição_{position}')

    column_stats = [
        {'column': column, 'position': position, 'expected_type': expected_schema.get(column), 'observed_types': Counter(), 'non_empty_count': 0, 'null_count': 0, 'blank_count': 0, 'whitespace_only_count': 0, 'type_error_count': 0, 'invalid_examples': []}
        for position, column in enumerate(header, start=1)
    ]
    row_width_errors = 0
    for row_number, row in enumerate(data_rows, start=2):
        if len(row) != len(header):
            row_width_errors += 1
            add_issue(issues, relative_path, row_number, 'row_width_mismatch', detail=f'esperado={len(header)}, encontrado={len(row)}')
        values = row[:len(header)] + [''] * max(0, len(header) - len(row))
        for index, value in enumerate(values):
            stats = column_stats[index]
            expected_type = stats['expected_type']
            value_type = infer_value_type(value)
            stats['observed_types'][value_type] += 1
            if value == '':
                stats['blank_count'] += 1
                add_issue(issues, relative_path, row_number, 'empty_string', column=stats['column'], value=value)
                continue
            if not value.strip():
                stats['whitespace_only_count'] += 1
                add_issue(issues, relative_path, row_number, 'whitespace_only_string', column=stats['column'], value=value)
                continue
            if is_null(value):
                stats['null_count'] += 1
                add_issue(issues, relative_path, row_number, 'null_value', column=stats['column'], value=value)
                continue
            stats['non_empty_count'] += 1
            type_error = (expected_type == 'number' and parse_number(value) is None) or (expected_type == 'url' and not is_valid_url(value))
            if type_error:
                stats['type_error_count'] += 1
                if len(stats['invalid_examples']) < 5:
                    stats['invalid_examples'].append(value[:120])
                issue_name = 'invalid_number' if expected_type == 'number' else 'invalid_url'
                add_issue(issues, relative_path, row_number, issue_name, column=stats['column'], value=value, detail=f'tipo_esperado={expected_type}')

    column_profiles = []
    for stats in column_stats:
        column_profiles.append({
            'file_path': relative_path,
            'column': stats['column'],
            'position': stats['position'],
            'expected_type': stats['expected_type'],
            'observed_types': dict(stats['observed_types']),
            'non_empty_count': stats['non_empty_count'],
            'null_count': stats['null_count'],
            'blank_count': stats['blank_count'],
            'whitespace_only_count': stats['whitespace_only_count'],
            'type_error_count': stats['type_error_count'],
            'invalid_examples': stats['invalid_examples'],
            'status': 'attention' if stats['null_count'] or stats['blank_count'] or stats['whitespace_only_count'] or stats['type_error_count'] else 'ok',
        })

    if decoded['encoding_status'] == 'attention':
        add_issue(issues, relative_path, None, 'encoding_attention', detail=f"encoding={decoded['encoding']}, replacements={decoded['replacement_count']}, controls={decoded['control_count']}")
    schema_issues = bool(missing_columns or unexpected_columns or duplicate_columns or blank_column_names or row_width_errors)
    report = {
        'file_path': relative_path,
        'rows': len(data_rows),
        'column_count': len(header),
        'column_names': header,
        'expected_columns': list(expected_schema),
        'missing_columns': missing_columns,
        'unexpected_columns': unexpected_columns,
        'duplicate_columns': duplicate_columns,
        'blank_column_names': blank_column_names,
        'encoding': decoded['encoding'],
        'encoding_status': decoded['encoding_status'],
        'delimiter': delimiter,
        'row_width_errors': row_width_errors,
        'schema_status': 'attention' if schema_issues else 'ok',
        'validation_status': 'attention' if issues else 'ok',
        'status': 'attention' if schema_issues or issues else 'ok',
    }
    return report, column_profiles, issues

In [4]:
file_inventory: list[dict] = []
schema_inventory: list[dict] = []
column_profile: list[dict] = []
validation_issues: list[dict] = []
dataset_summary: list[dict] = []

for dataset_name, config in DATASETS.items():
    root = config['root']
    dataset_files = inventory_files(dataset_name, root)
    file_inventory.extend(dataset_files)
    expected_by_filename = config.get('expected_schema_by_filename', {})
    tabular_paths = [
        root / row['relative_path']
        for row in dataset_files
        if row['tabular_candidate']
    ]
    dataset_reports = []
    for path in tabular_paths:
        report, profiles, issues = validate_tabular_file(
            path, root, expected_by_filename.get(path.name)
        )
        schema_inventory.append(report)
        column_profile.extend(profiles)
        validation_issues.extend(issues)
        dataset_reports.append(report)
    dataset_summary.append({
        'dataset': dataset_name,
        'root': str(root),
        'root_exists': root.is_dir(),
        'files_total': len(dataset_files),
        'tabular_files': len(tabular_paths),
        'files_by_suffix': dict(Counter(row['suffix'] for row in dataset_files)),
        'tabular_files_with_attention': sum(report['status'] == 'attention' for report in dataset_reports),
    })

len(file_inventory), len(schema_inventory), len(column_profile), len(validation_issues)

# Tradução apenas para apresentação; os nomes das colunas de origem permanecem intactos.
LABELS_PTBR = {
    'dataset': 'dataset',
    'root': 'raiz',
    'root_exists': 'raiz_existe',
    'relative_path': 'caminho_relativo',
    'file_path': 'arquivo',
    'suffix': 'extensão',
    'size_bytes': 'tamanho_bytes',
    'tabular_candidate': 'candidato_tabelar',
    'files_total': 'quantidade_arquivos',
    'tabular_files': 'quantidade_arquivos_tabelares',
    'files_by_suffix': 'arquivos_por_extensão',
    'tabular_files_with_attention': 'arquivos_tabelares_com_atenção',
    'rows': 'quantidade_linhas',
    'column_count': 'quantidade_colunas',
    'column_names': 'colunas',
    'expected_columns': 'colunas_esperadas',
    'missing_columns': 'colunas_ausentes',
    'unexpected_columns': 'colunas_inesperadas',
    'duplicate_columns': 'colunas_duplicadas',
    'blank_column_names': 'nomes_de_coluna_vazios',
    'encoding': 'codificação',
    'encoding_status': 'status_da_codificação',
    'delimiter': 'delimitador',
    'row_width_errors': 'linhas_com_quantidade_de_campos_incorreta',
    'schema_status': 'status_do_esquema',
    'validation_status': 'status_da_validação',
    'status': 'status_geral',
    'column': 'coluna',
    'position': 'posição',
    'expected_type': 'tipo_esperado',
    'observed_types': 'tipos_observados',
    'non_empty_count': 'quantidade_preenchidos',
    'null_count': 'quantidade_nulos',
    'blank_count': 'quantidade_strings_vazias',
    'whitespace_only_count': 'quantidade_apenas_espaços',
    'type_error_count': 'quantidade_erros_de_tipo',
    'invalid_examples': 'exemplos_inválidos',
    'row_number': 'linha',
    'issue': 'problema',
    'value_sample': 'amostra_do_valor',
    'detail': 'detalhe',
    'total_issues': 'total_de_problemas',
    'issues_by_type': 'problemas_por_tipo',
}
TYPE_LABELS_PTBR = {'string': 'texto', 'number': 'número', 'url': 'URL', 'empty': 'vazio', 'null': 'nulo', 'boolean': 'booleano'}
STATUS_LABELS_PTBR = {'ok': 'ok', 'attention': 'atenção'}
ISSUE_LABELS_PTBR = {
    'encoding_undecodable': 'codificação não interpretável',
    'encoding_attention': 'atenção na codificação',
    'csv_parse_error': 'erro de leitura CSV',
    'empty_tabular_file': 'arquivo tabular vazio',
    'duplicate_column': 'coluna duplicada',
    'blank_column_name': 'nome de coluna vazio',
    'row_width_mismatch': 'quantidade de campos inconsistente',
    'empty_string': 'string vazia',
    'whitespace_only_string': 'string somente com espaços',
    'null_value': 'valor nulo',
    'invalid_number': 'número não interpretável',
    'invalid_url': 'URL inválida',
}

def translate_record(record: dict) -> dict:
    translated = {}
    for key, value in record.items():
        output_key = LABELS_PTBR.get(key, key)
        if key == 'expected_type':
            value = TYPE_LABELS_PTBR.get(value, value)
        elif key == 'observed_types':
            value = {TYPE_LABELS_PTBR.get(name, name): count for name, count in value.items()}
        elif key == 'issue':
            value = ISSUE_LABELS_PTBR.get(value, value)
        elif key == 'issues_by_type':
            value = {ISSUE_LABELS_PTBR.get(name, name): count for name, count in value.items()}
        elif key in {'status', 'schema_status', 'validation_status', 'encoding_status'}:
            value = STATUS_LABELS_PTBR.get(value, value)
        translated[output_key] = value
    return translated

dataset_summary_df = pd.DataFrame([translate_record(record) for record in dataset_summary])
file_inventory_df = pd.DataFrame([translate_record(record) for record in file_inventory])
schema_inventory_df = pd.DataFrame([translate_record(record) for record in schema_inventory])
column_profile_df = pd.DataFrame([translate_record(record) for record in column_profile])
validation_issues_df = pd.DataFrame([translate_record(record) for record in validation_issues])
issues_summary_df = pd.DataFrame([translate_record({
    'total_issues': len(validation_issues),
    'issues_by_type': dict(Counter(issue['issue'] for issue in validation_issues)),
})])

## Quais arquivos existem?

A tabela mostra um resumo do que foi encontrado no diretório do dataset. A lista completa está em `file_inventory_df`.

Arquivos `.mp4` são vídeos e não são tratados como tabelas. Arquivos `.csv` e `.tsv` são os que entram na conferência de colunas. Se aparecerem arquivos `.metadata` ou `.incomplete`, eles pertencem ao cache do download e ficam registrados, mas não são analisados como dados.

In [5]:
dataset_summary_df

,dataset,raiz,raiz_existe,quantidade_arquivos,quantidade_arquivos_tabelares,arquivos_por_extensão,arquivos_tabelares_com_atenção
0,v-librasil-raw,c:\Users\mdbaa\development\Dissertacao\app\dat...,True,8189,2,"{'<sem_extensão>': 2, '.tag': 1, '.metadata': ...",0


## Quais colunas cada arquivo tem?

Cada linha representa um arquivo tabular. A coluna `colunas` mostra o cabeçalho exatamente como foi encontrado no arquivo.

- `colunas_ausentes` indica o que era esperado, mas não apareceu.
- `colunas_inesperadas` indica campos adicionais que não estavam previstos.
- `colunas_duplicadas` indica nomes repetidos no cabeçalho.
- `status_do_esquema = ok` significa que o cabeçalho e a estrutura das linhas estão consistentes.

In [6]:
schema_inventory_df

,arquivo,quantidade_linhas,quantidade_colunas,colunas,colunas_esperadas,colunas_ausentes,colunas_inesperadas,colunas_duplicadas,nomes_de_coluna_vazios,codificação,status_da_codificação,delimitador,linhas_com_quantidade_de_campos_incorreta,status_do_esquema,status_da_validação,status_geral
0,annotations.csv,4086,9,"[video_id, video_name, class, user_id, width, ...","[video_id, video_name, class, user_id, width, ...",[],[],[],[],utf-8-sig,ok,",",0,ok,ok,ok
1,error.csv,3,9,"[video_id, video_name, class, user_id, width, ...","[video_id, video_name, class, user_id, width, ...",[],[],[],[],utf-8-sig,ok,",",0,ok,ok,ok


## Os valores estão preenchidos e no formato esperado?

Cada linha do perfil representa uma coluna de um arquivo. Use estas colunas para entender o resultado:

- `tipo_esperado` mostra o formato que estamos esperando.
- `tipos_observados` mostra o que foi encontrado nos valores.
- `quantidade_nulos` e `quantidade_strings_vazias` mostram campos sem conteúdo.
- `quantidade_erros_de_tipo` mostra números ou URLs que não puderam ser interpretados.
- `status_geral = ok` indica que não foi encontrado problema naquela coluna.

Quando houver problemas, eles aparecem com mais detalhes em `validation_issues_df`, incluindo o arquivo, a linha e uma amostra do valor.

In [7]:
column_profile_df

,arquivo,coluna,posição,tipo_esperado,tipos_observados,quantidade_preenchidos,quantidade_nulos,quantidade_strings_vazias,quantidade_apenas_espaços,quantidade_erros_de_tipo,exemplos_inválidos,status_geral
0,annotations.csv,video_id,1,texto,{'texto': 4086},4086,0,0,0,0,[],ok
1,annotations.csv,video_name,2,texto,{'texto': 4086},4086,0,0,0,0,[],ok
2,annotations.csv,class,3,texto,{'texto': 4086},4086,0,0,0,0,[],ok
3,annotations.csv,user_id,4,texto,{'texto': 4086},4086,0,0,0,0,[],ok
4,annotations.csv,width,5,número,{'número': 4086},4086,0,0,0,0,[],ok
5,annotations.csv,height,6,número,{'número': 4086},4086,0,0,0,0,[],ok
6,annotations.csv,fps,7,número,{'número': 4086},4086,0,0,0,0,[],ok
7,annotations.csv,url_page,8,URL,{'URL': 4086},4086,0,0,0,0,[],ok
8,annotations.csv,url_download,9,URL,{'URL': 4086},4086,0,0,0,0,[],ok
9,error.csv,video_id,1,texto,{'texto': 3},3,0,0,0,0,[],ok


# Validação das chaves e da unicidade

Agora verificamos se os registros podem ser identificados sem ambiguidade. Esta etapa procura valores repetidos e conflitos entre as colunas, mas não decide se a quantidade de exemplos por classe é boa ou ruim.

Serão conferidos: `video_id`, `video_name`, `url_download`, a associação entre `video_id` e `class`, e repetições da combinação `class + user_id`.

In [8]:
def load_integrity_records(dataset_name: str, config: dict) -> list[dict]:
    root = config['root']
    records = []
    for item in file_inventory:
        if item['dataset'] != dataset_name or not item['tabular_candidate']:
            continue
        path = root / item['relative_path']
        decoded = decode_bytes(path.read_bytes())
        if decoded['text'] is None:
            continue
        _, dialect = detect_dialect(decoded['text'], path.suffix.lower())
        reader = csv.DictReader(io.StringIO(decoded['text']), dialect=dialect)
        for line_number, record in enumerate(reader, start=2):
            clean_record = {key: value for key, value in record.items() if key is not None}
            clean_record['_arquivo'] = item['relative_path']
            clean_record['_dataset'] = dataset_name
            clean_record['_linha'] = line_number
            records.append(clean_record)
    return records


integrity_records = []
for dataset_name, config in DATASETS.items():
    integrity_records.extend(load_integrity_records(dataset_name, config))
integrity_df = pd.DataFrame(integrity_records)

integrity_summary_rows = []
integrity_detail_rows = []


def valid_key_rows(columns: list[str]) -> pd.DataFrame:
    if integrity_df.empty or any(column not in integrity_df.columns for column in columns):
        return pd.DataFrame()
    valid = integrity_df.copy()
    mask = pd.Series(True, index=valid.index)
    for column in columns:
        mask &= valid[column].notna() & valid[column].astype(str).str.strip().ne('')
    return valid.loc[mask]


def check_repeated_key(label: str, columns: list[str], note: str) -> None:
    valid = valid_key_rows(columns)
    if valid.empty:
        integrity_summary_rows.append({
            'verificação': label,
            'colunas_usadas': ' + '.join(columns),
            'quantidade_grupos_repetidos': 0,
            'quantidade_linhas_envolvidas': 0,
            'status': 'não verificado',
            'observação': 'Coluna ausente ou sem valores preenchidos.',
        })
        return
    grouped = valid.groupby(columns, dropna=False, sort=True).agg(
        quantidade_linhas=('_linha', 'size'),
        arquivos=('_arquivo', lambda values: ', '.join(sorted(set(values)))),
    ).reset_index()
    repeated = grouped[grouped['quantidade_linhas'] > 1].copy()
    integrity_summary_rows.append({
        'verificação': label,
        'colunas_usadas': ' + '.join(columns),
        'quantidade_grupos_repetidos': len(repeated),
        'quantidade_linhas_envolvidas': int(repeated['quantidade_linhas'].sum()),
        'status': 'ok' if repeated.empty else 'atenção',
        'observação': note,
    })
    for _, row in repeated.iterrows():
        key = ' | '.join(f'{column}={row[column]}' for column in columns)
        integrity_detail_rows.append({
            'verificação': label,
            'chave': key,
            'quantidade_linhas': int(row['quantidade_linhas']),
            'arquivos': row['arquivos'],
            'observação': note,
        })


check_repeated_key('video_id repetido', ['video_id'], 'Duas ou mais linhas apontam para o mesmo identificador.')
check_repeated_key('video_name repetido', ['video_name'], 'Duas ou mais linhas usam o mesmo nome de vídeo.')
check_repeated_key('mesmo arquivo em url_download', ['url_download'], 'Duas ou mais linhas apontam para a mesma URL de download.')
check_repeated_key('combinação class + user_id repetida', ['class', 'user_id'], 'A repetição é registrada para investigação; não é classificada como boa ou ruim nesta etapa.')

valid = valid_key_rows(['video_id', 'class'])
if valid.empty:
    integrity_summary_rows.append({
        'verificação': 'video_id associado a classes diferentes',
        'colunas_usadas': 'video_id + class',
        'quantidade_grupos_repetidos': 0,
        'quantidade_linhas_envolvidas': 0,
        'status': 'não verificado',
        'observação': 'Coluna ausente ou sem valores preenchidos.',
    })
else:
    classes_por_video = valid.groupby('video_id', sort=True).agg(
        quantidade_classes=('class', 'nunique'),
        classes=('class', lambda values: ', '.join(sorted(set(values)))),
        quantidade_linhas=('class', 'size'),
    ).reset_index()
    conflicts = classes_por_video[classes_por_video['quantidade_classes'] > 1]
    integrity_summary_rows.append({
        'verificação': 'video_id associado a classes diferentes',
        'colunas_usadas': 'video_id + class',
        'quantidade_grupos_repetidos': len(conflicts),
        'quantidade_linhas_envolvidas': int(conflicts['quantidade_linhas'].sum()),
        'status': 'ok' if conflicts.empty else 'atenção',
        'observação': 'Cada video_id deve apontar para uma única classe.',
    })
    for _, row in conflicts.iterrows():
        integrity_detail_rows.append({
            'verificação': 'video_id associado a classes diferentes',
            'chave': f"video_id={row['video_id']}",
            'quantidade_linhas': int(row['quantidade_linhas']),
            'arquivos': '',
            'observação': f"classes encontradas: {row['classes']}",
        })

chaves_unicidade_df = pd.DataFrame(integrity_summary_rows)
conflitos_integridade_df = pd.DataFrame(integrity_detail_rows)

## Resultado das verificações

`chaves_unicidade_df` resume cada teste. O status `ok` significa que não houve repetição ou conflito naquele teste. O status `atenção` indica que existem linhas para investigar.

`conflitos_integridade_df` mostra quais chaves foram repetidas ou associadas a mais de uma classe.

In [9]:
chaves_unicidade_df

,verificação,colunas_usadas,quantidade_grupos_repetidos,quantidade_linhas_envolvidas,status,observação
0,video_id repetido,video_id,0,0,ok,Duas ou mais linhas apontam para o mesmo ident...
1,video_name repetido,video_name,0,0,ok,Duas ou mais linhas usam o mesmo nome de vídeo.
2,mesmo arquivo em url_download,url_download,0,0,ok,Duas ou mais linhas apontam para a mesma URL d...
3,combinação class + user_id repetida,class + user_id,0,0,ok,A repetição é registrada para investigação; nã...
4,video_id associado a classes diferentes,video_id + class,0,0,ok,Cada video_id deve apontar para uma única classe.


In [10]:
conflitos_integridade_df

""


# Validação CSV × filesystem

Aqui comparamos o que está registrado nos CSVs com o que realmente existe na pasta `videos/`. A comparação usa `video_name` como nome do arquivo.

Os arquivos listados em `error.csv` são tratados como exceções explícitas: eles não entram como ausentes ou não registrados. Um arquivo presente, mas com estrutura MP4 inválida ou tamanho zero, aparece separadamente como inválido ou corrompido.

In [11]:
def inspect_video_file(path: Path) -> tuple[bool, str]:
    try:
        size = path.stat().st_size
        if size == 0:
            return False, 'arquivo vazio'
        with path.open('rb') as stream:
            head = stream.read(1024 * 1024)
            stream.seek(max(size - 1024 * 1024, 0))
            tail = stream.read(1024 * 1024)
        sample = head + tail
        if b'ftyp' not in head[:256]:
            return False, 'assinatura MP4 ftyp ausente'
        if b'moov' not in sample:
            return False, 'estrutura MP4 moov ausente'
        if b'mdat' not in sample:
            return False, 'estrutura MP4 mdat ausente'
        return True, 'estrutura MP4 mínima encontrada'
    except OSError as error:
        return False, f'erro ao ler arquivo: {error}'


video_health_rows = []
for dataset_name, config in DATASETS.items():
    video_dir = config['root'] / 'videos'
    video_paths = sorted(path for path in video_dir.rglob('*') if path.is_file()) if video_dir.is_dir() else []
    for path in video_paths:
        valid, reason = inspect_video_file(path)
        video_health_rows.append({
            'dataset': dataset_name,
            'video_name': path.name,
            'caminho': path.relative_to(config['root']).as_posix(),
            'tamanho_bytes': path.stat().st_size,
            'arquivo_valido': valid,
            'motivo': reason,
        })
video_health_df = pd.DataFrame(video_health_rows)

filesystem_summary_rows = []
filesystem_detail_rows = []
filesystem_exception_rows = []

for dataset_name, config in DATASETS.items():
    dataset_records = integrity_df[integrity_df['_dataset'] == dataset_name] if not integrity_df.empty else pd.DataFrame()
    annotations_records = dataset_records[dataset_records['_arquivo'] == 'annotations.csv'] if not dataset_records.empty else pd.DataFrame()
    error_records = dataset_records[dataset_records['_arquivo'] == 'error.csv'] if not dataset_records.empty else pd.DataFrame()
    annotations_names = set(annotations_records['video_name'].dropna().astype(str).str.strip()) if 'video_name' in annotations_records else set()
    error_names = set(error_records['video_name'].dropna().astype(str).str.strip()) if 'video_name' in error_records else set()
    physical_records = video_health_df[video_health_df['dataset'] == dataset_name]
    physical_names = set(physical_records['video_name'])
    valid_names = set(physical_records.loc[physical_records['arquivo_valido'], 'video_name'])
    invalid_names = set(physical_records.loc[~physical_records['arquivo_valido'], 'video_name'])
    expected_names = annotations_names - error_names
    registered_existing = expected_names & valid_names
    registered_missing = expected_names - physical_names
    existing_unregistered = valid_names - annotations_names - error_names

    categories = [
        ('registrados e existentes', registered_existing, 'video_name aparece no annotations.csv e existe como arquivo válido.'),
        ('registrados mas ausentes', registered_missing, 'video_name aparece no annotations.csv, mas não existe em videos/.'),
        ('existentes mas não registrados', existing_unregistered, 'arquivo válido existe em videos/, mas não aparece no annotations.csv nem no error.csv.'),
        ('inválidos ou corrompidos', invalid_names, 'arquivo existe, mas falhou na verificação estrutural mínima do MP4.'),
    ]
    for category, names, explanation in categories:
        filesystem_summary_rows.append({
            'dataset': dataset_name,
            'categoria': category,
            'quantidade': len(names),
            'observação': explanation,
        })
        for name in sorted(names):
            filesystem_detail_rows.append({
                'dataset': dataset_name,
                'categoria': category,
                'video_name': name,
            })
    for name in sorted(error_names):
        filesystem_exception_rows.append({
            'dataset': dataset_name,
            'video_name': name,
            'origem': 'error.csv',
            'situação_física': 'presente em videos/' if name in physical_names else 'ausente em videos/',
            'observação': 'Exceção explícita: não entra nos conjuntos de ausência ou não registro.',
        })

filesystem_reconciliation_df = pd.DataFrame(filesystem_summary_rows)
filesystem_reconciliation_details_df = pd.DataFrame(filesystem_detail_rows)
filesystem_error_exceptions_df = pd.DataFrame(filesystem_exception_rows)

## Resultado da comparação

A tabela abaixo contém os quatro números principais. Para uma coincidência perfeita, a maior parte deve estar em `registrados e existentes`, enquanto as outras categorias devem ficar zeradas, salvo exceções documentadas em `error.csv`.

Os nomes individuais encontrados em cada categoria ficam em `filesystem_reconciliation_details_df`. Os nomes listados no `error.csv` ficam separados em `filesystem_error_exceptions_df`.

In [12]:
filesystem_reconciliation_df

,dataset,categoria,quantidade,observação
0,v-librasil-raw,registrados e existentes,4077,video_name aparece no annotations.csv e existe...
1,v-librasil-raw,registrados mas ausentes,9,"video_name aparece no annotations.csv, mas não..."
2,v-librasil-raw,existentes mas não registrados,9,"arquivo válido existe em videos/, mas não apar..."
3,v-librasil-raw,inválidos ou corrompidos,0,"arquivo existe, mas falhou na verificação estr..."


In [13]:
filesystem_reconciliation_details_df

,dataset,categoria,video_name
0,v-librasil-raw,registrados e existentes,Abacaxi_Articulador1.mp4
1,v-librasil-raw,registrados e existentes,Abacaxi_Articulador2.mp4
2,v-librasil-raw,registrados e existentes,Abacaxi_Articulador3.mp4
3,v-librasil-raw,registrados e existentes,Abanar_Articulador1.mp4
4,v-librasil-raw,registrados e existentes,Abanar_Articulador2.mp4
...,...,...,...
4090,v-librasil-raw,existentes mas não registrados,FrenteÁ Frente_Articulador2.mp4
4091,v-librasil-raw,existentes mas não registrados,FrenteÁ Frente_Articulador3.mp4
4092,v-librasil-raw,existentes mas não registrados,O quê_Articulador1.mp4
4093,v-librasil-raw,existentes mas não registrados,O quê_Articulador2.mp4


In [14]:
filesystem_error_exceptions_df

,dataset,video_name,origem,situação_física,observação
0,v-librasil-raw,Congelar_Articulador3.mp4,error.csv,ausente em videos/,Exceção explícita: não entra nos conjuntos de ...
1,v-librasil-raw,Criança_Articulador3.mp4,error.csv,ausente em videos/,Exceção explícita: não entra nos conjuntos de ...
2,v-librasil-raw,Poesia_Articulador2.mp4,error.csv,ausente em videos/,Exceção explícita: não entra nos conjuntos de ...


# Validação dos metadados técnicos CSV × vídeo real

Esta etapa chama a validação reutilizável dentro do Docker de processamento. Ela compara `width`, `height` e `fps` do CSV com os valores observados pelo `ffprobe`, sem alterar os arquivos em `data/raw/`.

A saída JSON é a fonte de verdade; os CSVs derivados são carregados para inspeção no notebook.

In [15]:
import json
import subprocess

METADATA_DATASET = 'v-librasil-raw'
METADATA_OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'metadata-validation' / METADATA_DATASET
METADATA_COMPOSE_FILE = PROJECT_ROOT / 'docker' / 'video-processing' / 'compose.yaml'
RUN_METADATA_VALIDATION = True

def run_metadata_validation_in_docker() -> None:
    command = [
        'docker', 'compose', '-f', str(METADATA_COMPOSE_FILE),
        'run', '--rm', '-T', 'video-processing',
        'python', '-m', 'libras_translation.cli',
        'video', 'validate-metadata', '--dataset', METADATA_DATASET,
    ]
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, shell=False, capture_output=True,
        text=True, check=False, encoding='utf-8', errors='replace',
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f'Validação de metadados falhou com código {completed.returncode}.')

if RUN_METADATA_VALIDATION:
    run_metadata_validation_in_docker()

metadata_validation_json_path = METADATA_OUTPUT_ROOT / 'metadata_validation.json'
metadata_validation = json.loads(metadata_validation_json_path.read_text(encoding='utf-8'))
metadata_validation_summary_df = pd.DataFrame([metadata_validation['summary']])
metadata_video_df = pd.read_csv(METADATA_OUTPUT_ROOT / 'video_metadata.csv')
metadata_comparison_df = pd.read_csv(METADATA_OUTPUT_ROOT / 'metadata_comparison.csv')
fps_hypotheses_summary_df = pd.read_csv(METADATA_OUTPUT_ROOT / 'fps_hypotheses_summary.csv')

     Validação de metadados —     
          v-librasil-raw          
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Métrica           ┃ Quantidade ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Vídeos vistos     │       4092 │
│ Inspeções OK      │       4086 │
│ Erros de inspeção │          6 │
│ FPS como taxa     │          0 │
│ FPS como frames   │       4083 │
│ FPS inconclusivo  │          6 │
└───────────────────┴────────────┘
Resultados: /workspace/app/data/processed/metadata-validation/v-librasil-raw



## Resultado da validação técnica

`fps_rate_status` testa `csv.fps == actual_fps`; `fps_frame_count_status` testa `csv.fps == actual_frame_count`. A coluna original `fps` permanece intacta. `inconclusive` indica que um valor necessário não estava disponível; `none` indica que os dois valores confiáveis não coincidiram.

In [16]:
display(metadata_validation_summary_df)
display(fps_hypotheses_summary_df)
display(metadata_comparison_df.head())

,dataset,videos_seen,videos_ok,videos_partial,videos_with_probe_error,width_matches,height_matches,fps_rate_matches,fps_frame_count_matches,fps_none_matches,fps_inconclusive,fps_hypothesis_counts,dataset_fps_interpretation,fps_tolerance
0,v-librasil-raw,4092,4086,0,6,4083,4083,0,4083,0,6,"{'frame_count': 4083, 'inconclusive': 6, 'not_...",frame_count,0.01


,hypothesis,count
0,frame_count,4083
1,inconclusive,6
2,not_in_csv,3


,relative_path,csv_width,csv_height,csv_fps_raw,csv_fps_numeric,width_status,height_status,fps_rate_status,fps_frame_count_status,fps_hypothesis,overall_status
0,Abacaxi_Articulador1.mp4,1920.0,1080.0,172.0,172.0,match,match,mismatch,match,frame_count,match
1,Abacaxi_Articulador2.mp4,1920.0,1080.0,135.0,135.0,match,match,mismatch,match,frame_count,match
2,Abacaxi_Articulador3.mp4,1920.0,1080.0,173.0,173.0,match,match,mismatch,match,frame_count,match
3,Abanar_Articulador1.mp4,1920.0,1080.0,407.0,407.0,match,match,mismatch,match,frame_count,match
4,Abanar_Articulador2.mp4,1920.0,1080.0,116.0,116.0,match,match,mismatch,match,frame_count,match


# Validação da convenção do nome do arquivo

O nome do arquivo deve carregar os mesmos metadados registrados no CSV. Para um arquivo como `Abacaxi_Articulador1.mp4`, o nome indica `class = Abacaxi` e `user_id = Articulador1`.

A comparação abaixo procura diferenças entre o que o nome informa e o que está em `annotations.csv` ou `error.csv`. Também verifica arquivos físicos que ainda não possuem uma linha no CSV.

In [17]:
def parse_video_filename(video_name: str) -> tuple[str | None, str | None, str | None]:
    path = Path(str(video_name))
    if path.suffix.casefold() != '.mp4':
        return None, None, 'extensão diferente de .mp4'
    if '_' not in path.stem:
        return None, None, 'nome não contém separador class_user_id'
    filename_class, filename_user = path.stem.rsplit('_', 1)
    if not filename_class or not re.fullmatch(r'Articulador\d+', filename_user):
        return filename_class or None, filename_user or None, 'user_id no nome não segue Articulador + número'
    return filename_class, filename_user, None


filename_convention_rows = []

for dataset_name in DATASETS:
    physical = video_health_df[video_health_df['dataset'] == dataset_name]
    for _, row in physical.iterrows():
        filename_class, filename_user, problem = parse_video_filename(row['video_name'])
        filename_convention_rows.append({
            'dataset': dataset_name,
            'origem': 'filesystem',
            'linha': None,
            'video_name': row['video_name'],
            'class_do_nome': filename_class,
            'user_id_do_nome': filename_user,
            'class_registrada': None,
            'user_id_registrado': None,
            'status': 'atenção' if problem else 'ok',
            'problema': problem or 'nome estruturalmente válido; sem registro CSV para comparar',
        })

    registered = integrity_df[(integrity_df['_dataset'] == dataset_name) & integrity_df['_arquivo'].isin(['annotations.csv', 'error.csv'])] if not integrity_df.empty else pd.DataFrame()
    for _, row in registered.iterrows():
        filename_class, filename_user, problem = parse_video_filename(row.get('video_name', ''))
        problems = []
        if problem:
            problems.append(problem)
        if not problem and filename_class != row.get('class'):
            problems.append(f"class registrada={row.get('class')!r}, class do nome={filename_class!r}")
        if not problem and filename_user != row.get('user_id'):
            problems.append(f"user_id registrado={row.get('user_id')!r}, user_id do nome={filename_user!r}")
        filename_convention_rows.append({
            'dataset': dataset_name,
            'origem': row['_arquivo'],
            'linha': row['_linha'],
            'video_name': row.get('video_name'),
            'class_do_nome': filename_class,
            'user_id_do_nome': filename_user,
            'class_registrada': row.get('class'),
            'user_id_registrado': row.get('user_id'),
            'status': 'atenção' if problems else 'ok',
            'problema': '; '.join(problems),
        })

filename_convention_df = pd.DataFrame(filename_convention_rows)
filename_convention_summary_df = (
    filename_convention_df.groupby(['origem', 'status'], dropna=False)
    .size()
    .reset_index(name='quantidade')
)

## Resultado da convenção dos nomes

`status = ok` significa que o nome segue a convenção e, quando há registro CSV, bate com `class` e `user_id`. As divergências aparecem em `filename_convention_df`, com o número da linha do CSV e os dois valores para comparação.

In [18]:
filename_convention_summary_df

,origem,status,quantidade
0,annotations.csv,atenção,6
1,annotations.csv,ok,4080
2,error.csv,ok,3
3,filesystem,ok,4086


In [19]:
filename_convention_df[filename_convention_df['status'] == 'atenção']

,dataset,origem,linha,video_name,class_do_nome,user_id_do_nome,class_registrada,user_id_registrado,status,problema
4817,v-librasil-raw,annotations.csv,733.0,"Cérebro (de ""ervilha"""")_Articulador3.mp4""",NaN,NaN,"Cérebro (de ""ervilha"""")""",Articulador3,atenção,extensão diferente de .mp4
4818,v-librasil-raw,annotations.csv,734.0,"Cérebro (de ""ervilha"""")_Articulador2.mp4""",NaN,NaN,"Cérebro (de ""ervilha"""")""",Articulador2,atenção,extensão diferente de .mp4
4819,v-librasil-raw,annotations.csv,735.0,"Cérebro (de ""ervilha"""")_Articulador1.mp4""",NaN,NaN,"Cérebro (de ""ervilha"""")""",Articulador1,atenção,extensão diferente de .mp4
5895,v-librasil-raw,annotations.csv,1811.0,Frente\Á Frente_Articulador3.mp4,Á Frente,Articulador3,Frente\Á Frente,Articulador3,atenção,"class registrada='Frente\\Á Frente', class do ..."
5896,v-librasil-raw,annotations.csv,1812.0,Frente\Á Frente_Articulador2.mp4,Á Frente,Articulador2,Frente\Á Frente,Articulador2,atenção,"class registrada='Frente\\Á Frente', class do ..."
5897,v-librasil-raw,annotations.csv,1813.0,Frente\Á Frente_Articulador1.mp4,Á Frente,Articulador1,Frente\Á Frente,Articulador1,atenção,"class registrada='Frente\\Á Frente', class do ..."


# Validação de proveniência, decisão e manifest derivado

Esta etapa vem depois da convenção de nomes. Ela reconcilia `error.csv` com `annotations.csv` e `videos/`, procura ausências silenciosas e produz evidências auditáveis. `GO` significa que não foram encontrados problemas `CRITICAL` ou `ERROR`; `NO-GO` não apaga nem altera o dataset original.

In [ ]:
QUALITY_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'metadata-validation'
QUALITY_REPORT_ROOT = QUALITY_ROOT / 'reports'
QUALITY_MANIFEST_ROOT = QUALITY_ROOT / 'manifests'

def run_quality_validation_in_docker() -> None:
    command = [
        'docker', 'compose', '-f', str(METADATA_COMPOSE_FILE),
        'run', '--rm', '-T', 'video-processing',
        'python', '-m', 'libras_translation.cli',
        'video', 'validate-quality', '--dataset', METADATA_DATASET,
    ]
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, shell=False, check=False,
        encoding='utf-8', errors='replace',
    )
    if completed.returncode not in (0, 2):
        raise RuntimeError(f'Geração dos artefatos de qualidade falhou com código {completed.returncode}.')

run_quality_validation_in_docker()
metadata_summary = json.loads((QUALITY_REPORT_ROOT / 'metadata_summary.json').read_text(encoding='utf-8'))
metadata_issues_df = pd.read_csv(QUALITY_REPORT_ROOT / 'metadata_issues.csv')
validated_manifest_df = pd.read_parquet(QUALITY_MANIFEST_ROOT / 'v_librasil_validated.parquet')
metadata_decision = metadata_summary['decision']

## Evidências da decisão

O resumo JSON e o CSV de issues são a fonte de evidência da decisão. O Parquet é um manifest derivado, com metadados observados e flags de uso; nenhum arquivo em `data/raw/` é sobrescrito.

In [ ]:
display(pd.DataFrame([{'dataset': metadata_summary['dataset'], 'decision': metadata_decision, **metadata_summary['counts']['by_severity']}]))
display(metadata_issues_df.head(50))
display(validated_manifest_df.head())

## Como interpretar o resultado

Se `status_geral`, `status_do_esquema` e `status_da_validação` estiverem como `ok`, o arquivo foi lido e os campos encontrados estão no formato básico esperado.

Isso ainda não garante que o vídeo seja válido para treinamento. Aqui verificamos se os metadados podem ser lidos e se os registros não apresentam conflitos básicos de identificação. Regras como dimensões positivas, FPS plausível, vídeo corrompido e distribuição das classes ficam para etapas posteriores.